# Input Space Mode Connectivity — Walkthrough

**"Input Space Mode Connectivity in Deep Neural Networks"**  
Vrabel, Shem-Ur, Oz, Krueger — ICLR 2025 — [paper](https://openreview.net/forum?id=3qeOy7HwUT)

Four experiments:
1. **Sec 4.1** — Connectivity between real images (GoogLeNet / ImageNet)
2. **Sec 4.2** — Adversarial attacks and barrier statistics
3. **Sec 4.3** — Connectivity in untrained models (synthetic FVO inputs)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from utils import *

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Section 4.1 — Input space connectivity (real images)

Two low-loss ImageNet images of the same class are linearly interpolated.
A loss barrier appears; we optimize it away (Figures 1–3).

In [ ]:
model = load_googlenet(device)

In [ ]:
# Class 955 = jackfruit — same images as in Figure 1
CLASS_ID = 955
TENSOR_DIR = "./data_sel/saved_tensors"

x_a, x_c = load_saved_tensors(TENSOR_DIR, CLASS_ID, (0, 1), device)
plot_images([x_a, x_c], labels=["A", "C"])
plt.show()

In [ ]:
# Linear interpolation A → C (1000 steps)
alphas, points = linear_interpolation(x_a, x_c, steps=1000)
losses = compute_losses(model, points, CLASS_ID)

idx, alpha_max, barrier = find_barrier(alphas, points, losses)
print(f"Barrier at α={alpha_max:.3f}, loss={losses[idx]:.4f}")

plot_loss_curve(alphas, losses, title="Primary A → C (jackfruit)")
plt.show()

In [ ]:
# Show A, B (barrier point), C
plot_images(
    [x_a, barrier, x_c],
    labels=["A", "B (barrier)", "C"],
    losses=[losses[0], losses[idx], losses[-1]],
)
plt.show()

In [ ]:
# Optimize barrier B → B'
b_prime, _ = bypass_barrier(
    model, x_a, x_c, barrier, CLASS_ID,
    lr=0.005, iterations=1024,
    lambda_mse=0.1, lambda_hf=1e-8, verbose=True,
)

In [ ]:
# Evaluate the bypass path A→B'→C
_, pts_ab = linear_interpolation(x_a, b_prime, steps=500)
losses_ab = compute_losses(model, pts_ab, CLASS_ID)

_, pts_bc = linear_interpolation(b_prime, x_c, steps=500)
losses_bc = compute_losses(model, pts_bc, CLASS_ID)

print(f"Max loss A→B': {losses_ab.max():.6f}")
print(f"Max loss B'→C: {losses_bc.max():.6f}")

plot_barrier_comparison(
    alphas, losses, losses_ab, losses_bc, alpha_max,
    title="Jackfruit (955) — GoogLeNet",
)
plt.show()

### Try other classes / models

Saved tensors available: 574, 763, 954, 955.

```python
model = load_resnet18_imagenet(device)  # or load_vit(device)
x_a, x_c = load_saved_tensors(TENSOR_DIR, 574, (0, 1), device)  # golf ball
```

---
## Section 4.2 — Adversarial attacks and barrier statistics

An image from a different class is optimized (targeted attack) to minimize CE for the
target class, producing an adversarial example. The barrier between the original real
image and the adversarial example is then computed (Figures 4–5).

Pre-computed barrier statistics across all 1000 ImageNet classes are loaded and shown
as boxplots (Figure 6).

In [ ]:
from sec4_2_adversarial_attacks import create_adversarial

TARGET_CLASS = 574   # golf ball
SOURCE_CLASS = 763   # revolver
TENSOR_DIR = "./data_sel/saved_tensors"

# Real ImageNet images: golf ball (sel_1) and revolver (sel_1)
x_a = load_saved_tensors(TENSOR_DIR, TARGET_CLASS, (1,), device)[0]
x_k = load_saved_tensors(TENSOR_DIR, SOURCE_CLASS, (1,), device)[0]

# Create adversarial example: optimize x_k to minimize CE for TARGET_CLASS
x_k_adv, adv_loss = create_adversarial(
    model, x_k, TARGET_CLASS,
    lr=0.005, iterations=1024,
    lambda_mse=0.1, lambda_hf=1e-7, verbose=True,
)
print(f"Adversarial loss: {adv_loss:.5f}")

# Show A (golf ball), K (revolver), K' (adversarial)
loss_a = compute_losses(model, x_a.unsqueeze(0) if x_a.dim() == 3 else x_a, TARGET_CLASS)[0]
loss_k = compute_losses(model, x_k.unsqueeze(0) if x_k.dim() == 3 else x_k, TARGET_CLASS)[0]
plot_images(
    [x_a, x_k, x_k_adv],
    labels=["A (golf ball)", "K (revolver)", "K' (adversarial)"],
    losses=[loss_a, loss_k, adv_loss],
)
plt.show()

# Barrier between A and K'
alphas_adv, pts_adv = linear_interpolation(x_a, x_k_adv, steps=250)
losses_adv = compute_losses(model, pts_adv, TARGET_CLASS)
idx_adv, alpha_adv_max, _ = find_barrier(alphas_adv, pts_adv, losses_adv)
print(f"Barrier at α={alpha_adv_max:.3f}, loss={losses_adv[idx_adv]:.4f}")

plot_loss_curve(alphas_adv, losses_adv,
                title="Barrier: real (golf ball) → adversarial")
plt.show()

### Barrier statistics across ImageNet classes (Figure 6)

The single-pair demo above illustrates the phenomenon for one (golf ball, revolver) pair.
Below we show barrier statistics aggregated over all 1 000 ImageNet classes —
comparing within-class barriers (two real images of the same class) against adversarial
barriers (real image vs. targeted adversarial example from a different class).

Pre-computed CSVs are loaded by default. To recompute from scratch (slow — runs over
all 1 000 classes × 7 pairs each), use `compute_within_class_barriers` and
`compute_adversarial_barriers` from `sec4_2_adversarial_attacks`, or run the CLI with
`python sec4_2_adversarial_attacks.py --compute --stat_tensors ./data_sel/stat_tensors`.

In [ ]:
import pandas as pd
from utils import load_googlenet
from sec4_2_adversarial_attacks import (
    compute_within_class_barriers, compute_adversarial_barriers,
    postprocess_barriers, plot_statistics,
)

DATA_DIR = "./data_sel"
COMPUTE_FROM_SCRATCH = False  # set True to recompute from scratch (~hours on GPU)

if COMPUTE_FROM_SCRATCH:
    stat_tensor_dir = f"{DATA_DIR}/stat_tensors"
    stat_model = load_googlenet(device)
    within = postprocess_barriers(compute_within_class_barriers(stat_model, stat_tensor_dir))
    cross = postprocess_barriers(compute_adversarial_barriers(stat_model, stat_tensor_dir))
else:
    within = pd.read_csv(f"{DATA_DIR}/interpolation_processed.csv")
    cross = pd.read_csv(f"{DATA_DIR}/cross_class_interpolation_processed.csv")

plot_statistics(within, cross)

---
## Section 4.3 — Connectivity in untrained models

Synthetic optimal inputs via FVO on a randomly initialized ResNet-18.
Connectivity exists even without training (Figure 7).


In [ ]:
# Sec 4.3 setup — self-contained, matches random_network.ipynb exactly.
# Run this cell first when starting Sec 4.3 only. Order: model → seed (no random ops before model).
import torch
import numpy as np
import matplotlib.pyplot as plt
from utils import (
    load_resnet18_untrained,
    linear_interpolation, compute_losses, find_barrier,
    optimize_point, feature_visualization,
    plot_loss_curve, plot_three_rounds,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model first (no seed), then seed — exact order as random_network.ipynb
model_untrained = load_resnet18_untrained(device)
TARGET = 1  # CIFAR-10 class index

seed = 0
torch.manual_seed(seed)

In [ ]:
# Generate both tensors sequentially (seed set in setup cell; no re-seeding between A and C).
# Tensor A: with hf penalty
x_a_syn, la = feature_visualization(
    model_untrained, TARGET, lr=0.05, iterations=8192,
    loss_threshold=0.0005, lambda_hf=2.5e-7,
    noise_scale=0.01, weight_decay=1e-7, device=device, verbose=True,
)
print(f"Loss A: {la:.6f}")

# Tensor C: no penalty — inherits random state after A (no re-seeding).
x_c_syn, lc = feature_visualization(
    model_untrained, TARGET, lr=0.05, iterations=8192,
    loss_threshold=0.0005, lambda_hf=0.0,
    noise_scale=0.01, weight_decay=1e-7, device=device, verbose=True,
)
print(f"Loss C: {lc:.6f}")

In [ ]:
# Primary interpolation A → C
N = 200
alphas_p, pts_p = linear_interpolation(x_a_syn, x_c_syn, steps=N)
losses_p = compute_losses(model_untrained, pts_p, TARGET)
idx_p, _, bp_tensor = find_barrier(alphas_p, pts_p, losses_p)
print(f"Primary barrier: loss={losses_p[idx_p]:.4f}")

plot_loss_curve(alphas_p, losses_p, title="Primary (untrained ResNet-18)")
plt.show()

In [ ]:
# --- Round 1: optimize the primary barrier point B → B' ---
# Free (unconstrained) optimization of the max-loss point to minimize CE.
print("Round 1: optimizing primary barrier...")
b_prime, bl = optimize_point(
    model_untrained, bp_tensor, TARGET,
    lr=0.05, iterations=4096, weight_decay=1e-7,
    loss_threshold=0.0005, verbose=True,
)
print(f"B' loss: {bl:.5f}")

# Secondary interpolation: A→B' and B'→C
a1, p1 = linear_interpolation(x_a_syn, b_prime, steps=N)
a2, p2 = linear_interpolation(b_prime, x_c_syn, steps=N)
l1 = compute_losses(model_untrained, p1, TARGET)
l2 = compute_losses(model_untrained, p2, TARGET)

# --- Round 2: optimize each secondary barrier ---
def bypass_segment(start, end, losses_seg, seg_name=""):
    """Find max-loss point in a segment and optimize it."""
    _, pts_seg = linear_interpolation(start, end, steps=N)
    idx_seg = int(np.argmax(losses_seg))
    print(f"  {seg_name} barrier loss={losses_seg[idx_seg]:.4f}")
    opt, ol = optimize_point(
        model_untrained, pts_seg[idx_seg], TARGET,
        lr=0.05, iterations=4096, weight_decay=1e-7,
        loss_threshold=0.0005,
    )
    print(f"  optimized to {ol:.5f}")
    _, pa = linear_interpolation(start, opt, steps=N)
    _, pb = linear_interpolation(opt, end, steps=N)
    return compute_losses(model_untrained, pa, TARGET), compute_losses(model_untrained, pb, TARGET)

print("\nRound 2: optimizing secondary barriers...")
l1_1, l1_2 = bypass_segment(x_a_syn, b_prime, l1, "A→B'")
l2_1, l2_2 = bypass_segment(b_prime, x_c_syn, l2, "B'→C")

# 3-panel plot matching Figure 7
plot_three_rounds(
    alphas_p, losses_p,
    [a1, a2], [l1, l2],
    [None, None, None, None], [l1_1, l1_2, l2_1, l2_2],
    title="Untrained ResNet-18 — synthetic inputs (Sec 4.3, Fig. 7)",
)
plt.show()

The ternary plot should show that after two rounds, the loss along the 4-segment path is much lower than the original primary barrier. Additional rounds can further reduce remaining bumps.

---
## Standalone scripts

Each experiment can also be run from the command line:

```bash
python sec4_1_input_space_connectivity.py --model googlenet --class_id 955
python sec4_2_adversarial_attacks.py --target_class 574 --source_class 763
python sec4_2_adversarial_attacks.py --compute --stat_tensors ./data_sel/stat_tensors  # recompute stats from scratch
python sec4_3_untrained_model.py --target_class 1
```